# RandomDataStreams.jl — a tour

This notebook runs the package end to end: the stream/substream model, the same
code on all four generator families, the addressing property that only
counter-based generators have, and the reason the package exists — common random
numbers.

Everything here is executable and the outputs below were produced by running it.
The reference documentation is in [`docs/`](../docs/src/index.md); this notebook is
the hands-on counterpart.


In [1]:
using Pkg
# works whether the notebook is launched from the repository root or from notebooks/
Pkg.activate(isfile("Project.toml") ? "." : "..")

using RandomDataStreams, Random, Statistics

  Activating project at `~/Git/RandomDataStreams.jl`


## 1. Streams and substreams

A *generator object* hands out *streams*; each stream is divided into
*substreams*. Nothing is shared between two streams, and navigation is explicit:

| call | effect |
|---|---|
| `next_stream!(gen)` | a new, independent stream |
| `next_substream!(rng)` | move to the start of the next substream |
| `reset_substream!(rng)` | back to the start of the current substream |
| `reset_stream!(rng)` | back to the very beginning of the stream |

The seed here is fixed so that the notebook is reproducible.


In [2]:
gen  = MRG32k3aGen([1, 2, 3, 4, 5, 6])
rngA = next_stream!(gen)
rngB = next_stream!(gen)

show(rngA)

Full state of MRG32k3a generator:
Cg = [1, 2, 3, 4, 5, 6]
Bg = [1, 2, 3, 4, 5, 6]
Ig = [1, 2, 3, 4, 5, 6]

`reset_substream!` replays the substream you are in, exactly:


In [3]:
first_pass  = [rand(rngA) for _ in 1:5]
reset_substream!(rngA)
second_pass = [rand(rngA) for _ in 1:5]

first_pass == second_pass

true

Substream boundaries are *anchored*: `next_substream!` lands in the same place
no matter how much of the current substream you consumed. This is what makes a
replication reproducible even when the model under study changes how many random
numbers it draws.


In [4]:
reset_stream!(rngA)
next_substream!(rngA)
from_the_start = [rand(rngA) for _ in 1:5]

reset_stream!(rngA)
for _ in 1:37; rand(rngA); end      # consume an arbitrary amount
next_substream!(rngA)
after_consuming = [rand(rngA) for _ in 1:5]

from_the_start == after_consuming

true

Two streams from the same generator never overlap:


In [5]:
reset_stream!(rngA)
isempty(intersect([rand(rngA) for _ in 1:1000], [rand(rngB) for _ in 1:1000]))

true

## 2. Every navigation call, on every class of generator

Section 1 used MRG32k3a to keep the narrative readable. The package's claim,
though, is that the *same* calls mean the *same* thing on every design it
ships, so it is worth running them all against one representative of each
class:

| Representative | Class | A stream is |
|---|---|---|
| `MRG32k3a` | combined multiple recursive generator | a segment of one orbit |
| `Xoshiro256pp` | F2-linear recurrence | a segment of one orbit |
| `PCG64` | linear congruential + output permutation | a segment of one orbit |
| `Philox4x64-10` | counter-based, wide multiply | a key |
| `Threefry4x64-20` | counter-based, add–rotate–xor | a key |

Each column below is a property the interface promises. Nothing in the
function knows which generator it is driving.


In [6]:
families = ["MRG32k3a"        => () -> MRG32k3aGen(20260830),
            "Xoshiro256pp"    => () -> Xoshiro256ppGen(20260830),
            "PCG64"           => () -> PCG64Gen(20260830),
            "Philox4x64-10"   => () -> Philox4x64Gen(20260830),
            "Threefry4x64-20" => () -> Threefry4x64Gen(20260830)]

function navigation_report(makegen)
    gen  = makegen()
    a, b = next_stream!(gen), next_stream!(gen)   # next_stream!

    # two streams of the same generator share nothing
    streams_disjoint = isempty(intersect([rand(a) for _ in 1:1000],
                                         [rand(b) for _ in 1:1000]))

    # reset_stream! rewinds to the very beginning, whatever was consumed
    reset_stream!(a)
    from_start = [rand(a) for _ in 1:5]
    for _ in 1:3; next_substream!(a); end
    for _ in 1:40; rand(a); end
    reset_stream!(a)
    stream_rewinds = [rand(a) for _ in 1:5] == from_start

    # next_substream! is anchored: it does not matter how much was consumed
    reset_stream!(a); next_substream!(a)
    substream = [rand(a) for _ in 1:5]
    reset_stream!(a)
    for _ in 1:37; rand(a); end
    next_substream!(a)
    substream_anchored = [rand(a) for _ in 1:5] == substream

    # reset_substream! replays the substream you are in
    reset_substream!(a)
    substream_replays = [rand(a) for _ in 1:5] == substream

    # and two substreams of one stream are disjoint too
    reset_stream!(a)
    s1 = [rand(a) for _ in 1:1000]
    next_substream!(a)
    s2 = [rand(a) for _ in 1:1000]
    substreams_disjoint = isempty(intersect(s1, s2))

    return (streams_disjoint, stream_rewinds, substream_anchored,
            substream_replays, substreams_disjoint)
end

cols = ["streams", "reset_stream!", "next_substream!", "reset_substream!", "substreams"]
println(rpad("generator", 18), join(lpad.(cols, 18)))
for (name, makegen) in families
    println(rpad(name, 18), join(lpad.(navigation_report(makegen), 18)))
end

generator                    streams     reset_stream!   next_substream!  reset_substream!        substreams
MRG32k3a                        true              true              true              true              true
Xoshiro256pp                    true              true              true              true              true
PCG64                           true              true              true              true              true
Philox4x64-10                   true              true              true              true              true
Threefry4x64-20                 true              true              true              true              true


The full `Random` API works on all of them — not just `rand()`:


In [7]:
rng = next_stream!(Philox4x64Gen())

println(rand(rng, UInt64))
println(rand(rng, 1:6, 5))
println(round.(randn(rng, 3), digits = 4))
println(shuffle(rng, collect(1:8)))

1609277786247541068
[6, 6, 3, 1, 2]
[-0.4267, 1.1914, -1.8007]
[3, 4, 7, 2, 1, 5, 8, 6]


## 3. Counter-based generators: a stream is a key

Recurrence-based generators reach position *i* by *getting there* — you jump along
the orbit. A counter-based generator computes the draw at position *i* directly:
the block is a keyed bijection of the counter, so `(key, substream, index)` is an
address, not a state.

Below, the same four words are obtained twice: once by navigating a stream object,
and once by building a generator straight at that address, with nothing replayed.
This is what lets a GPU kernel or a remote worker reproduce a stream the host
assigned it, having received only the key.


In [8]:
gen = PhiloxGen()
rng = next_stream!(gen)
key = get_state(rng)[2]              # this stream's key

for _ in 1:3
    next_substream!(rng)             # walk to substream 3
end
navigated = [rand(rng, UInt32) for _ in 1:4]

# the counter splits in half: the high 64 bits index the substream
direct = PhiloxRNG(collect(key), UInt128(3) << 64)
recomputed = [rand(direct, UInt32) for _ in 1:4]

println("key        = ", key)
println("navigated  = ", navigated)
println("recomputed = ", recomputed)
navigated == recomputed

key        = (0x00000000, 0x00000000)
navigated  = UInt32[0xf0a90abc, 0x115896b8, 0xbedefe84, 0xb86f45c6]
recomputed = UInt32[0xf0a90abc, 0x115896b8, 0xbedefe84, 0xb86f45c6]


true

## 4. Common random numbers

This is what the stream model is for. We compare two order quantities in a
newsvendor problem: demand is exponential with mean 100, the item sells for 10 and
costs 6, and we want $E[\text{profit}(q_2)] - E[\text{profit}(q_1)]$.

One replication = one substream. With **common random numbers**, replication *r*
of both configurations uses the *same* substream, so the two systems see identical
demand and the difference isolates the effect of the decision. With independent
streams, the demand noise is counted twice.


In [9]:
const price, cost, mean_demand = 10.0, 6.0, 100.0

demand(u)    = -mean_demand * log1p(-u)          # inverse CDF of Exp(1/100)
profit(q, d) = price * min(q, d) - cost * q

function crn_difference(gen, q1, q2, nreps)
    rng = next_stream!(gen)                      # ONE stream, one substream per replication
    d = zeros(nreps)
    for r in 1:nreps
        dem  = demand(rand(rng))                 # the same demand for both quantities
        d[r] = profit(q2, dem) - profit(q1, dem)
        next_substream!(rng)
    end
    return d
end

function independent_difference(gen, q1, q2, nreps)
    r1, r2 = next_stream!(gen), next_stream!(gen)   # a separate stream per configuration
    d = zeros(nreps)
    for r in 1:nreps
        d[r] = profit(q2, demand(rand(r2))) - profit(q1, demand(rand(r1)))
        next_substream!(r1); next_substream!(r2)
    end
    return d
end

independent_difference (generic function with 1 method)

In [10]:
q1, q2, nreps = 90.0, 110.0, 20_000

dc = crn_difference(MRG32k3aGen([1, 2, 3, 4, 5, 6]), q1, q2, nreps)
di = independent_difference(MRG32k3aGen([1, 2, 3, 4, 5, 6]), q1, q2, nreps)

exact(q) = price * mean_demand * (1 - exp(-q / mean_demand)) - cost * q

for (label, d) in ("common random numbers" => dc, "independent streams" => di)
    hw = 1.96 * std(d) / sqrt(nreps)
    println(rpad(label, 24), "estimate = ", lpad(round(mean(d), digits = 2), 8),
            "   +/- ", round(hw, digits = 2))
end
println(rpad("exact value", 24), "         = ", lpad(round(exact(q2) - exact(q1), digits = 2), 8))
println("\nvariance ratio = ", round(var(di) / var(dc), digits = 1), "x")

common random numbers   estimate =   -47.03   +/- 1.3
independent streams     estimate =   -40.71   +/- 7.03
exact value                      =    -46.3

variance ratio = 29.3x


Both estimators are unbiased and both cover the exact value. The difference is
the width of the interval: common random numbers cut the variance by a factor of
about 30, which is a factor of 5–6 on the confidence interval — the same precision
for roughly 1/30th of the replications.

Nothing about this depends on the generator. Substituting `Philox4x64Gen()` or a
xoshiro generator above changes the numbers, not the conclusion.


## 5. Saving, restoring and jumping — again on every class

`get_state` and `set_state!` are exact inverses everywhere, and
`advance_state!(rng, e, c)` moves `2^e + c` draws in either direction, with
one draw as the unit for every generator. What differs is only the cost: a
sequence of matrix products for MRG32k3a, GF(2) polynomial arithmetic for
xoshiro, `O(log n)` multiplications for PCG, and a single counter addition for
the counter-based families.


In [11]:
function state_report(makegen)
    r = next_stream!(makegen())

    # get_state / set_state! round-trip exactly
    saved    = get_state(r)
    expected = [rand(r) for _ in 1:5]
    set_state!(r, saved)
    restores = [rand(r) for _ in 1:5] == expected

    # ... and restoring does not disturb the substream anchor
    set_state!(r, saved)
    reset_substream!(r)
    v0 = rand(r)
    reset_substream!(r)
    anchor_kept = rand(r) == v0

    # advance_state! counts draws, forwards ...
    a = next_stream!(makegen())
    base = [rand(a) for _ in 1:12]
    b = next_stream!(makegen())
    advance_state!(b, 0, 5)
    forwards = [rand(b) for _ in 1:3] == base[6:8]

    # ... and backwards
    advance_state!(b, 0, -8)
    backwards = [rand(b) for _ in 1:3] == base[1:3]

    return (restores, anchor_kept, forwards, backwards)
end

cols = ["set_state!", "anchor kept", "advance +", "advance -"]
println(rpad("generator", 18), join(lpad.(cols, 15)))
for (name, makegen) in families
    println(rpad(name, 18), join(lpad.(state_report(makegen), 15)))
end

generator              set_state!    anchor kept      advance +      advance -
MRG32k3a                     true           true           true           true
Xoshiro256pp                 true           true           true           true
PCG64                        true           true           true           true
Philox4x64-10                true           true           true           true
Threefry4x64-20              true           true           true           true


## Where to go next

- [Getting started](../docs/src/getting_started.md) — installation and the basics
- [Streams & substreams](../docs/src/streams.md) — the model, and the host/device boundary
- [Generator comparison](../docs/src/comparison.md) — which generator, and why
- [Validation](../docs/src/validation.md) — TestU01 batteries, and how to run the heavy ones
- [Implementation notes](../docs/src/implementation.md) — algorithms and measured throughput
